# Organização do Ambiente

Neste notebook preparamos o ambiente do projeto. Primeiro importamos as configurações globais, criamos o catálogo `v_credit`, os schemas `bronze`, `silver` e `gold` e o Volume onde os arquivos brutos serão armazenados. No final, validamos de forma rigorosa se todos os arquivos obrigatórios do case foram enviados.

In [0]:
%run ./00_Config

## Criação da infraestrutura

Usamos `IF NOT EXISTS` para que qualquer integrante do grupo consiga executar esta etapa sem tentar recriar objetos que já existem, garantindo idempotência na infraestrutura.

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.{landing_volume_name}")

print(f"Catálogo: {catalog}")
print(f"Schemas: {bronze_schema}, {silver_schema}, {gold_schema}")
print(f"Landing Zone: {landing_path}")

## Validação dos arquivos de entrada

Antes de seguir para a camada Bronze, verificamos se os oito arquivos esperados estão presentes no Volume. Caso algum arquivo esteja faltando, a execução é interrompida imediatamente com uma listagem clara dos pendentes.

In [0]:
expected_files = {
    "base_atendentes.csv", "base_motivos.csv", "canais.csv",
    "chamados.csv", "Chamados_Hora.CSV", "clientes.csv",
    "custos.csv", "pesquisa_satisfacao.csv",
}

existing_files = {file.name for file in dbutils.fs.ls(landing_path)}
missing_files = sorted(expected_files - existing_files)
unexpected_files = sorted(existing_files - expected_files)

assert not missing_files, (
    "Landing Zone incompleta. Faça upload de: " + ", ".join(missing_files)
)

print(f"[OK] {len(expected_files)} arquivos obrigatórios encontrados.")
if unexpected_files:
    print(f"[AVISO] Arquivos adicionais serão ignorados: {unexpected_files}")